## High level analysis

- Number of job adverts
- Number of job adverts per occupation
- Number of job adverts per region
- Salaries
- Outliers

In [37]:
import pandas as pd
from dap_prinz_green_jobs.getters.data_getters import load_s3_data
from dap_prinz_green_jobs import BUCKET_NAME, analysis_config
from dap_prinz_green_jobs.utils.chloropleth_utils import get_nuts2polygons_dict, get_nuts1polygons_dict, get_nuts3polygons_dict
from dap_prinz_green_jobs.getters.industry_getters import load_sic
import dap_prinz_green_jobs.analysis.ojo_analysis.process_ojo_green_measures as pg

import altair as alt
import ast

In [38]:
occ_agg_gje = pd.read_csv(
    f"s3://prinz-green-jobs/outputs/data/ojo_application/extracted_green_measures/analysis/occupation_aggregated_data_{analysis_config['analysis_files']['agg_soc_date_stamp']}_extra_gjeformat.csv")

len(occ_agg_gje)

1059

In [39]:
occ_agg_gje['num_job_ads'].min()

51

In [40]:
occ_agg = pd.read_csv(
    f"s3://prinz-green-jobs/outputs/data/ojo_application/extracted_green_measures/analysis/occupation_aggregated_data_{analysis_config['analysis_files']['agg_soc_date_stamp']}_all.csv")

occ_agg = occ_agg[occ_agg['clean_soc_name']!='Betting shop managers']
occ_agg.reset_index(inplace=True)

In [41]:
itl_aggregated_data = pd.DataFrame()
for agg_itl_by in ["itl_1_code", "itl_2_code", "itl_3_code"]:
    date_stamp = analysis_config['analysis_files']['agg_region_date_stamp']
    per_itl_aggregated_data = load_s3_data(
        BUCKET_NAME,
        f"outputs/data/ojo_application/extracted_green_measures/analysis/{agg_itl_by}_aggregated_data_{date_stamp}.csv"
        )

    per_itl_aggregated_data['average_perc_green_skills'] = per_itl_aggregated_data['average_prop_green_skills']*100
    per_itl_aggregated_data['average_prop_occ_green_timeshare'] = per_itl_aggregated_data['average_occ_green_timeshare']/100

    per_itl_aggregated_data.loc[:, "itl_type"] = agg_itl_by
    per_itl_aggregated_data.rename(columns = {agg_itl_by: "itl_code"}, inplace=True)
    per_itl_aggregated_data.drop([f"{agg_itl_by}.1"], axis=1, inplace=True)
    itl_aggregated_data = pd.concat([itl_aggregated_data, per_itl_aggregated_data])

In [42]:
nuts1polygons_dict = get_nuts1polygons_dict()
itl1polygons_dict = {k.replace("UK","TL"):v for k, v in nuts1polygons_dict.items()}

nuts2polygons_dict = get_nuts2polygons_dict()
itl2polygons_dict = {k.replace("UK","TL"):v for k, v in nuts2polygons_dict.items()}

nuts3polygons_dict = get_nuts3polygons_dict()
itl3polygons_dict = {k.replace("UK","TL"):v for k, v in nuts3polygons_dict.items()}

allpolygons_dict = {**itl1polygons_dict, **itl2polygons_dict, **itl3polygons_dict}

In [43]:
# Just using this to get the ITL names
itl_aggregated_data['geometry_name'] = itl_aggregated_data["itl_code"].map(allpolygons_dict)
itl_aggregated_data[['geometry', 'itl_name']] = itl_aggregated_data['geometry_name'].apply(lambda x: pd.Series(x))
itl_aggregated_data.drop('geometry_name', axis=1, inplace=True)
itl_aggregated_data.drop('geometry', axis=1, inplace=True)

In [44]:
itl1_aggregated_data = itl_aggregated_data[itl_aggregated_data['itl_type']=='itl_1_code']
itl2_aggregated_data = itl_aggregated_data[itl_aggregated_data['itl_type']=='itl_2_code']
itl3_aggregated_data = itl_aggregated_data[itl_aggregated_data['itl_type']=='itl_3_code']

## Occupations

In [45]:
print(f"There are {occ_agg['num_job_ads'].sum()} job adverts assigned to {len(occ_agg)} SOC 6-digit occupations")
print(f"These are from {occ_agg['SOC_2020'].nunique()} unique SOC 4-digit codes")
print(f"There are {sum(occ_agg['num_job_ads']>50)} ({round(sum(occ_agg['num_job_ads']>50)*100/len(occ_agg),2)})% SOC 6-digit occupations with over 50 job adverts")
print(f"There are {sum(occ_agg['num_job_ads']>100)} ({round(sum(occ_agg['num_job_ads']>100)*100/len(occ_agg),2)})% SOC 6-digit occupations with over 100 job adverts")

There are 3922478 job adverts assigned to 1325 SOC 6-digit occupations
These are from 408 unique SOC 4-digit codes
There are 1059 (79.92)% SOC 6-digit occupations with over 50 job adverts
There are 955 (72.08)% SOC 6-digit occupations with over 100 job adverts


In [46]:
print("In the GJE..")
print(f"There are {occ_agg_gje['num_job_ads'].sum()} job adverts assigned to {len(occ_agg_gje)} SOC 6-digit occupations")
print(f"These are from {occ_agg_gje['SOC_2020'].nunique()} unique SOC 4-digit codes")

In the GJE..
There are 3917638 job adverts assigned to 1059 SOC 6-digit occupations
These are from 407 unique SOC 4-digit codes


In [47]:
thresh = 50
a = alt.Chart(
    occ_agg[occ_agg['num_job_ads']<=thresh], title=f"Number of job adverts <= {thresh}"
         ).mark_bar(color='blue').encode(
    alt.X("num_job_ads", title= "Number of job adverts (binned)", bin=True),
    y=alt.Y('count()', title="Number of occupations"),
)

b = alt.Chart(
    occ_agg[occ_agg['num_job_ads']>thresh], title=f"Number of job adverts > {thresh}",
         ).mark_bar(color='blue').encode(
    alt.X("num_job_ads", title= "Number of job adverts (binned)", bin=alt.BinParams(maxbins=40)),
    y=alt.Y('count()', title="Number of occupations").scale(type="log"),
)

a |b

alt.HConcatChart(...)

In [48]:
occ_agg[occ_agg['num_job_ads']>50]['num_job_ads'].quantile(.25)

215.0

In [49]:
occ_agg[occ_agg['num_job_ads']>50]['num_job_ads'].median()

772.0

In [50]:
occ_agg[occ_agg['num_job_ads']>50]['num_job_ads'].quantile(.5)

772.0

In [51]:
occ_agg[occ_agg['num_job_ads']>50]['num_job_ads'].quantile(.75)

2987.5

In [52]:
sum(occ_agg['num_job_ads'].between(50,200))/sum(occ_agg['num_job_ads']>50)

0.23418319169027385

In [53]:
sum(occ_agg['num_job_ads'].between(50,5000))/sum(occ_agg['num_job_ads']>50)

0.8253068932955618

In [54]:
print(occ_agg[occ_agg['num_job_ads']==1]['clean_soc_name'].tolist())

['Alexander technique teachers', 'Senior police officers', 'Agricultural contractors', 'Bin cleaners', 'Gunsmiths', 'Performance make-up artists', 'Film and television runners', 'Falconers', 'Homeopaths (excludes medically qualified)', 'Fishers', 'Body piercers', 'Forestry and related workers', 'Careers advisers and vocational guidance specialists', 'Thatchers', 'Milliners (excludes wholesale, retail trade)', 'Non-commissioned Royal Navy officers and other ranks', 'Authors', 'Monumental masons', 'Divers', 'Livery yard and stud farm managers and proprietors', 'Reflexologists', 'Hypnotherapists', 'Weight loss advisers', 'Antenatal teachers', 'Drain cleaners', 'Fish and river keepers', 'Estate agents and auctioneers']


In [55]:
occ_agg[occ_agg['num_job_ads']<=50].sort_values(by='num_job_ads')[['clean_soc_name', 'num_job_ads']][0:10]

,clean_soc_name,num_job_ads
1324,Estate agents and auctioneers,1
1245,Gunsmiths,1
1270,Film and television runners,1
1228,Bin cleaners,1
1277,Falconers,1
1216,Agricultural contractors,1
1284,Homeopaths (excludes medically qualified),1
1290,Fishers,1
1299,Body piercers,1
1300,Forestry and related workers,1


In [56]:
num_thresh = []
for thresh in range(0,1000):
    num_thresh.append({
        "Minimum number of job adverts": thresh,
        "Number of occupations":sum(occ_agg['num_job_ads']>thresh),
        "Proportion of occupations":sum(occ_agg['num_job_ads']>thresh)/len(occ_agg)
    })

In [57]:
alt.Chart(
    pd.DataFrame(num_thresh)
         ).mark_line(color='blue').encode(
    alt.X("Minimum number of job adverts"),
    y=alt.Y('Proportion of occupations'),
)

alt.Chart(...)

# Salaries
- In data for GjE

In [58]:
occ_agg_gje['median_min_annualised_salary'].max()

110000.0

In [59]:
occ_agg_gje[occ_agg_gje['median_min_annualised_salary']>100000][['clean_soc_name', 'median_min_annualised_salary']]

,clean_soc_name,median_min_annualised_salary
799,Educational psychologists,104000.0
1039,General practitioners,110000.0


In [60]:
occ_agg_sal_filt = occ_agg_gje[occ_agg_gje['median_min_annualised_salary']<=110000]
occ_agg_sal_filt['binned_median_min_annualised_salary'] = pd.cut(occ_agg_sal_filt['median_min_annualised_salary'], bins=10)

dd = occ_agg_sal_filt.groupby(['binned_median_min_annualised_salary'])['num_job_ads'].sum().reset_index()
dd['binned_median_min_annualised_salary_str'] = dd['binned_median_min_annualised_salary'].astype(str)
alt.Chart(dd[['num_job_ads', 'binned_median_min_annualised_salary_str']]).mark_bar(color='blue').encode(
    x=alt.X('num_job_ads', title="Number of job adverts"), # .scale(type="symlog")
    y=alt.Y('binned_median_min_annualised_salary_str', title="Minimum salary (binned)",sort=None),
    tooltip=[
        alt.Tooltip("num_job_ads", title="Number of job adverts"), 
    ]  
)

alt.Chart(...)

In [62]:
ddj = occ_agg_sal_filt.groupby(['binned_median_min_annualised_salary'])['SOC_2020_EXT_name'].nunique().reset_index()
ddj['binned_median_min_annualised_salary_str'] = ddj['binned_median_min_annualised_salary'].astype(str)
alt.Chart(ddj[['SOC_2020_EXT_name', 'binned_median_min_annualised_salary_str']]).mark_bar(color='blue').encode(
    x=alt.X('SOC_2020_EXT_name', title="Number of occupations"), # .scale(type="symlog")
    y=alt.Y('binned_median_min_annualised_salary_str', title="Minimum salary (binned)",sort=None),
    tooltip=[
        alt.Tooltip("SOC_2020_EXT_name", title="Number of occupations"), 
    ]  
)

alt.Chart(...)

## Number of skills
- Do some occupations have very small numbers of skills?

In [63]:
occ_agg_gje['average_prop_green_skills'].median()

0.0069264069264069

In [64]:
occ_agg_gje['average_num_skills'].median()

12.870056497175142

In [65]:
occ_agg_gje['average_num_skills'].quantile(0.25)

9.15923661376613

In [66]:
occ_agg_gje['average_num_skills'].quantile(0.75)

15.957734703529548

In [67]:
thresh = 5
print(f"There are {len(occ_agg_gje[occ_agg_gje['average_num_skills']<thresh])} occupations with less than an average of {thresh} skills per job advert")

print("There occupations are..")
print(occ_agg_gje[occ_agg_gje['average_num_skills']<thresh]['SOC_2020_EXT_name'].unique().tolist())

There are 17 occupations with less than an average of 5 skills per job advert
There occupations are..
['Smart energy experts', 'Mobile machine drivers and operatives n.e.c.', "Builder's labourers", 'Dockers, slingers and stevedores ', 'Bricklayers', 'Scaffolders and stagers', 'Groundworkers   ', 'Dry liners', 'Neonatal nurses', 'Steel fixers and underpinners', 'Directors in logistics, warehousing and transport n.e.c.', 'Podiatrists', 'Ceiling fitters', 'Surgeons ', 'Anaesthetists', 'Bicycle and motorcycle couriers', 'Mystery shoppers']


In [68]:
occ_agg_gje[occ_agg_gje['average_num_skills']<thresh][['SOC_2020_EXT_name', 'average_prop_green_skills']]

,SOC_2020_EXT_name,average_prop_green_skills
118,Smart energy experts,0.030317
145,Mobile machine drivers and operatives n.e.c.,0.026568
223,Builder's labourers,0.018952
263,"Dockers, slingers and stevedores",0.017009
290,Bricklayers,0.015534
297,Scaffolders and stagers,0.015165
373,Groundworkers,0.011634
481,Dry liners,0.007677
718,Neonatal nurses,0.003314
726,Steel fixers and underpinners,0.003205


In [69]:

occ_agg_gje['binned_average_num_skills'] = pd.cut(occ_agg_gje['average_num_skills'], bins=10)

dd = occ_agg_gje.groupby(['binned_average_num_skills'])['SOC_2020_EXT_name'].nunique().reset_index()
dd['binned_average_num_skills_str'] = dd['binned_average_num_skills'].astype(str)
alt.Chart(dd[['SOC_2020_EXT_name', 'binned_average_num_skills_str']]).mark_bar(color='blue').encode(
    x=alt.X('SOC_2020_EXT_name', title="Number of occupations"), # .scale(type="symlog")
    y=alt.Y('binned_average_num_skills_str', title="Average number of skills",sort=None),
    tooltip=[
        alt.Tooltip("SOC_2020_EXT_name", title="Number of occupations"), 
    ]  
)

alt.Chart(...)

## Regions

- ITL 1 and 2 all have over 100 job adverts

In [71]:
print(f"There are {itl1_aggregated_data['num_job_ads'].sum()} job adverts assigned to {itl1_aggregated_data['itl_code'].nunique()} ITL1 regions")
print(f"There are {itl2_aggregated_data['num_job_ads'].sum()} job adverts assigned to {itl2_aggregated_data['itl_code'].nunique()} ITL2 regions")
print(f"There are {itl3_aggregated_data['num_job_ads'].sum()} job adverts assigned to {itl3_aggregated_data['itl_code'].nunique()} ITL3 regions")

There are 4522288 job adverts assigned to 12 ITL1 regions
There are 4482509 job adverts assigned to 37 ITL2 regions
There are 4482509 job adverts assigned to 159 ITL3 regions


In [72]:
print(f"{sum(itl1_aggregated_data['num_job_ads']>1000)/len(itl1_aggregated_data)} ITL 1 regions have over 1000 job adverts")
print(f"{sum(itl2_aggregated_data['num_job_ads']>1000)/len(itl2_aggregated_data)} ITL 2 regions have over 1000 job adverts")
print(f"{sum(itl3_aggregated_data['num_job_ads']>1000)/len(itl3_aggregated_data)} ITL 3 regions have over 1000 job adverts")
print(f"{sum(itl3_aggregated_data['num_job_ads']>100)/len(itl3_aggregated_data)} ITL 3 regions have over 100 job adverts")

1.0 ITL 1 regions have over 1000 job adverts
1.0 ITL 2 regions have over 1000 job adverts
0.9119496855345912 ITL 3 regions have over 1000 job adverts
1.0 ITL 3 regions have over 100 job adverts


In [73]:
itl3_aggregated_data.sort_values(by='num_job_ads')[0:10][['itl_name', 'num_job_ads']]

,itl_name,num_job_ads
157,Darlington,106
146,Shetland Islands,345
117,Na h-Eileanan Siar (Western Isles),364
154,Causeway Coast and Glens,403
152,Orkney Islands,431
144,Ards and North Down,520
139,Mid and East Antrim,551
156,Antrim and Newtownabbey,553
115,Derry City and Strabane,554
158,Lisburn and Castlereagh,597


In [74]:
num_thresh_itl3 = []
for thresh in range(0,1000):
    num_thresh_itl3.append({
        "Minimum number of job adverts": thresh,
        "Number of regions":sum(itl3_aggregated_data['num_job_ads']>thresh),
        "Proportion of regions":sum(itl3_aggregated_data['num_job_ads']>thresh)/len(itl3_aggregated_data)
    })

alt.Chart(
    pd.DataFrame(num_thresh_itl3)
         ).mark_line(color='blue').encode(
    alt.X("Minimum number of job adverts"),
    y=alt.Y('Proportion of regions'),
)

alt.Chart(...)

In [75]:
alt.Chart(
    itl1_aggregated_data.sort_values(by="num_job_ads", ascending=False)
         ).mark_bar(color='blue').encode(
    x=alt.X('num_job_ads', title="Number of job adverts"),
    y = alt.Y("itl_name", title="ITL 1 region name", sort=None),
    tooltip=[
        alt.Tooltip("itl_name", title="Region"),
        alt.Tooltip("num_job_ads", title="Number of job adverts"),
        alt.Tooltip("prop_job_ads", title="Percentage of job adverts", format=".2%"),   
    ]  
)

alt.Chart(...)

In [76]:
alt.Chart(
    itl2_aggregated_data.sort_values(by="num_job_ads", ascending=False)
         ).mark_bar(color='blue').encode(
    x=alt.X('num_job_ads', title="Number of job adverts"),
    y = alt.Y("itl_name", title="ITL 2 region name", sort=None),
    tooltip=[
        alt.Tooltip("itl_name", title="Region"),
        alt.Tooltip("num_job_ads", title="Number of job adverts"),
        alt.Tooltip("prop_job_ads", title="Percentage of job adverts", format=".2%"),   
    ]  
)

alt.Chart(...)